# Mean fitting and interval efficiency

Discovery must cross-validate the same fitting path that it returns. This study tests a correction that makes non-Gaussian stopping use the existing fitting rate (the step length `nu`) 0.02, rather than tuning at 0.1 and then fitting at 0.02. It also establishes sharp-mean controls for distinguishing mean approximation error from varying noise scale.

The [Poisson protocol](poisson_tuning_consistency/PROTOCOL.md) was committed before implementation or measurement. Ten development and 50 independent confirmation draws per count regime use 1,000 analysis rows, an independent 1,000-row test sample, and the same 500 fitting/500 calibration partition for every method. The five regimes are low, original and high Poisson rates, overdispersion and zero inflation. The primary reference is prespecified LightGBM CQR (conformalized quantile regression: quantile fits for the lower and upper ends, widened by a conformal correction). Real interval width remains the primary efficiency metric; common-support count size is reported separately. Mean coverage must lie in [0.88, 0.92] at nominal 0.90. The target is at most 1% excess, with point and one-sided upper-bound results distinguished.

In [1]:
from pathlib import Path
import json
import sys

import numpy as np
import polars as pl
from great_tables import GT, md

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "Cargo.toml").exists())
sys.path.insert(0, str(root / "python"))
sys.path.insert(0, str(root))
import mboostrs as mb
assert Path(mb.__file__).resolve().is_relative_to(root)

directory = root / "docs/book/benchmarks/poisson_tuning_consistency"
rows = json.loads((directory / "confirmation_summary.json").read_text())
changes = json.loads((directory / "confirmation_changes.json").read_text())
facts = json.loads((directory / "confirmation_validation.json").read_text())
sharp = json.loads((directory.parent / "sharp_mean_controls/summary.json").read_text())

In [2]:
def label(method):
    name = method.replace("baseline_discovery", "Discovery before")
    name = name.replace("corrected_discovery", "Discovery matched")
    name = name.replace("supplied_fixed", "Supplied fixed").replace("supplied_tuned", "Supplied tuned")
    return name.replace("_poisson_normalized", " / normalized").replace("_poisson_quantile", " / count quantile") \
               .replace("_absolute", " / absolute").replace("_", " ")

leaders = {}
for panel in facts["panels"]:
    eligible = [r for r in rows if r["panel"] == panel and r["coverage_eligible"] and not r["failures"]]
    for metric in ("width", "count_size", "true_mean_mse", "interval_score"):
        leaders[panel, metric] = min((r[metric] for r in eligible), default=None)

def number(record, key, uncertainty=False):
    if key not in record:
        return "unavailable"
    text = f"{record[key]:.3f}"
    if uncertainty:
        text += f" ± {record[key + '_se']:.3f}"
    if record["coverage_eligible"] and record[key] == leaders.get((record["panel"], key)):
        text = f"**{text}**"
    return text

In [3]:
headline = pl.DataFrame([{
    "Regime": r["panel"], "Method": label(r["method"]), "Width": number(r, "width", True),
    "Count size": number(r, "count_size", True), "Coverage": number(r, "coverage"),
    "Mean MSE": number(r, "true_mean_mse"), "Interval score": number(r, "interval_score"),
    "Failures": r["failures"]} for r in rows])
(GT(headline, groupname_col="Regime")
 .tab_header(title="Every method on the frozen count confirmation",
             subtitle="Means over 50 independent draws per regime; smaller widths, count size, MSE and score are better")
 .tab_spanner(label="Prediction sets", columns=["Width", "Count size", "Coverage"])
 .fmt_markdown(columns=["Width", "Count size", "Mean MSE", "Interval score"])
 .tab_source_note(md("Width is real length in count units; count size is cardinality after the same nonnegative-integer "
    "intersection for every method. Width/count entries show mean ± Monte Carlo SE across draws. Coverage, MSE "
    "and interval score are means; their SEs are retained in the summary. Mean MSE compares each reported point "
    "with the generating mean; CQR points are band midpoints. Bold marks descriptive minima among complete "
    "coverage-eligible rows; it does not select a new procedure. An incomplete method is unavailable, never averaged "
    "over only successful draws. Source: confirmation_summary.json; runners and summary files are listed below.")))

GT(_tbl_data=shape: (85, 8)
┌─────────────┬────────────┬────────────┬────────────┬──────────┬──────────┬────────────┬──────────┐
│ Regime      ┆ Method     ┆ Width      ┆ Count size ┆ Coverage ┆ Mean MSE ┆ Interval   ┆ Failures │
│ ---         ┆ ---        ┆ ---        ┆ ---        ┆ ---      ┆ ---      ┆ score      ┆ ---      │
│ str         ┆ str        ┆ str        ┆ str        ┆ str      ┆ str      ┆ ---        ┆ i64      │
│             ┆            ┆            ┆            ┆          ┆          ┆ str        ┆          │
╞═════════════╪════════════╪════════════╪════════════╪══════════╪══════════╪════════════╪══════════╡
│ low         ┆ Discovery  ┆ **1.448 ±  ┆ 1.700 ±    ┆ 0.912    ┆ 0.011    ┆ **2.338**  ┆ 0        │
│             ┆ before /   ┆ 0.007**    ┆ 0.023      ┆          ┆          ┆            ┆          │
│             ┆ absolute   ┆            ┆            ┆          ┆          ┆            ┆          │
│ low         ┆ Discovery  ┆ 1.504 ±    ┆ 1.680 ±    ┆ 0.908    ┆ 0.011    ┆ 2.373      ┆ 0        │
│             ┆ before /   ┆ 0.015      ┆ 0.026      ┆          ┆          ┆            ┆          │
│             ┆ normalized ┆            ┆            ┆          ┆          ┆            ┆          │
│ low         ┆ Discovery  ┆ 1.110 ±    ┆ 2.110 ±    ┆ 0.967    ┆ 0.011    ┆ 1.855      ┆ 0        │
│             ┆ before /   ┆ 0.019      ┆ 0.019      ┆          ┆          ┆            ┆          │
│             ┆ count      ┆            ┆            ┆          ┆          ┆            ┆          │
│             ┆ quant…     ┆            ┆            ┆          ┆          ┆            ┆          │
│ low         ┆ Discovery  ┆ 1.491 ±    ┆ 1.701 ±    ┆ 0.913    ┆ 0.009    ┆ 2.362      ┆ 0        │
│             ┆ matched /  ┆ 0.010      ┆ 0.023      ┆          ┆          ┆            ┆          │
│             ┆ absolute   ┆            ┆            ┆          ┆          ┆            ┆          │
│ low         ┆ Discovery  ┆ 1.612 ±    ┆ 1.682 ±    ┆ 0.908    ┆ 0.009    ┆ 2.446      ┆ 0        │
│             ┆ matched /  ┆ 0.023      ┆ 0.026      ┆          ┆          ┆            ┆          │
│             ┆ normalized ┆            ┆            ┆          ┆          ┆            ┆          │
│ …           ┆ …          ┆ …          ┆ …          ┆ …        ┆ …        ┆ …          ┆ …        │
│ zero_inflat ┆ catboost   ┆ 39.122 ±   ┆ 33.034 ±   ┆ 0.902    ┆ 23.503   ┆ 52.596     ┆ 0        │
│ ed          ┆ split      ┆ 0.263      ┆ 0.154      ┆          ┆          ┆            ┆          │
│ zero_inflat ┆ catboost   ┆ 30.718 ±   ┆ 28.618 ±   ┆ 0.901    ┆ 8.881    ┆ 40.893     ┆ 0        │
│ ed          ┆ cqr        ┆ 0.190      ┆ 0.128      ┆          ┆          ┆            ┆          │
│ zero_inflat ┆ lightgbm   ┆ 40.230 ±   ┆ 33.624 ±   ┆ 0.901    ┆ 26.949   ┆ 52.702     ┆ 0        │
│ ed          ┆ split      ┆ 0.266      ┆ 0.149      ┆          ┆          ┆            ┆          │
│ zero_inflat ┆ lightgbm   ┆ 28.047 ±   ┆ **27.102 ± ┆ 0.900    ┆ 7.959    ┆ 34.658     ┆ 0        │
│ ed          ┆ cqr        ┆ 0.167      ┆ 0.100**    ┆          ┆          ┆            ┆          │
│ zero_inflat ┆ sklearn    ┆ **27.026 ± ┆ 27.428 ±   ┆ 0.914    ┆ 6.803    ┆ **32.272** ┆ 0        │
│ ed          ┆ cqr        ┆ 0.086**    ┆ 0.081      ┆          ┆          ┆            ┆          │
└─────────────┴────────────┴────────────┴────────────┴──────────┴──────────┴────────────┴──────────┘, _body=<great_tables._gt_data.Body object at 0x10d3cae40>, _boxhead=Boxhead([ColInfo(var='Regime', type=<ColInfoTypeEnum.row_group: 3>, column_label='Regime', column_align='left', column_width=None), ColInfo(var='Method', type=<ColInfoTypeEnum.default: 1>, column_label='Method', column_align='left', column_width=None), ColInfo(var='Width', type=<ColInfoTypeEnum.default: 1>, column_label='Width', column_align='left', column_width=None), ColInfo(var='Count size', type=<ColInfoTypeEnum.default: 1>, column_label='Count size', column_ali

The high-rate comparison isolates the practical consequence of the path mismatch. At the original rate, better mean fitting does not establish the default width target. Low-rate LightGBM CQR is outside the coverage window, and the overdispersed discovery comparison is unavailable because one of its 50 selected models is unsupported by the standard-model count-score instrument. These exclusions remain in the table.

## Fitting paths

The data-only workflow uses `discover` without a supplied family or formula on proper-training rows only. The supplied fixed track retains `bols(x1)+bols(x2)`, Poisson, 100 updates and rate 0.1. A supplied tuned control uses five inner folds, rate 0.02, a 100-step starting grid and a 400-step cap. It is an explanatory control, not a data-only success. Absolute, normalized and central-count quantile scores reuse each fitted mean unchanged.

The corrected procedure passes the existing family-specific learning rate into both stopping and its cost preflight. Gaussian fitting remains at 0.1. Poisson gradients, the exact stopping criterion, term/family selection, maximum grids and default absolute calibration are unchanged. The following example separates fitting and calibration explicitly; no calibration response selects the model.

In [4]:
from benches.study_poisson_tuning_consistency import draw
from benches.study_discovery_calibration import partition

# The first high-rate confirmation draw: seed 6494100, its test sample at seed + 100000,
# and the protocol's 500 fitting / 500 calibration split.
seed = 6494100
analysis, _, _ = draw("high", seed)
test, _, _ = draw("high", seed + 100000)
fitting, held, partition_record = partition(analysis, seed)

# Discovery sees only the fitting rows. The absolute-residual score at level 0.9 is the
# recorded method corrected_discovery_absolute.
selected = mb.discover(fitting, "y", seed=seed)
band = mb.conformal_interval(selected.model, test.drop("y"), calibration=held,
                             score="absolute", level=0.9, extrapolation="clip")

example = pl.DataFrame({"x1": test["x1"].head(3), "Mean": band.fitted_mean[:3],
                        "Lower": band.lower[:3], "Upper": band.upper[:3]})
(GT(example)
 .tab_header(title="Calibrated discovery predictions, high rate, seed 6494100",
             subtitle="Count units; fitting and calibration rows are disjoint")
 .fmt_number(columns=["Mean", "Lower", "Upper"], decimals=4)
 .tab_source_note(f"Selected fit: {selected.family}, {selected.formula}, stop {selected.mstop}; "
                  f"absolute-score correction {band.quantile:.4f}. This is the input and the settings of the "
                  "example record shown under Stopping ties and evidence; it adds no confirmation draw."))

GT(_tbl_data=shape: (3, 4)
┌───────────┬───────────┬───────────┬───────────┐
│ x1        ┆ Mean      ┆ Lower     ┆ Upper     │
│ ---       ┆ ---       ┆ ---       ┆ ---       │
│ f64       ┆ f64       ┆ f64       ┆ f64       │
╞═══════════╪═══════════╪═══════════╪═══════════╡
│ -0.992768 ┆ 6.788359  ┆ 0.483311  ┆ 13.093407 │
│ 0.841415  ┆ 17.014727 ┆ 10.709679 ┆ 23.319775 │
│ 0.446326  ┆ 18.850536 ┆ 12.545488 ┆ 25.155584 │
└───────────┴───────────┴───────────┴───────────┘, _body=<great_tables._gt_data.Body object at 0x112fd0dd0>, _boxhead=Boxhead([ColInfo(var='x1', type=<ColInfoTypeEnum.default: 1>, column_label='x1', column_align='right', column_width=None), ColInfo(var='Mean', type=<ColInfoTypeEnum.default: 1>, column_label='Mean', column_align='right', column_width=None), ColInfo(var='Lower', type=<ColInfoTypeEnum.default: 1>, column_label='Lower', column_align='right', column_width=None), ColInfo(var='Upper', type=<ColInfoTypeEnum.default: 1>, column_label='Upper', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x12319c320>, _spanners=Spanners([]), _heading=Heading(title='Calibrated discovery predictions, high rate, seed 6494100', subtitle='Count units; fitting and calibration rows are disjoint', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x122d09310>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x115b13470>, _source_notes=['Selected fit: Poisson, bols(x1) + bols(x2), stop 145; absolute-score correction 6.3050. This is the input and the settings of the example record shown under Stopping ties and evidence; it adds no confirmation draw.'], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x115620f80>, _formats=[<great_tables._gt_data.FormatInfo object at 0x10d99fa40>], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_top_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_right_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_right_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_right_color=OptionsInfo(scss=True, category='table', type='value', value='#D3D3D3'), table_border_bottom_include=OptionsInfo(scss=False, category='t

## Prespecified efficiency target

For each draw, the percentage real-width excess is $100(W_m/W_r-1)$, where $W_m$ is a method's mean test width and $W_r$ is the prespecified reference width on that draw. The table reports the mean paired excess, its Monte Carlo SE and the one-sided 95% Student-t upper confidence limit across independent draws. A point estimate below 1% and an upper limit below 1% are different results. An ineligible reference permits descriptive contrasts but no target passage.

The five original untuned controls remain CatBoost split/CQR (500 trees, depth 6, rate 0.05), LightGBM split/CQR (300 trees, 15 leaves, rate 0.05), and sklearn CQR (300 trees, depth 3, rate 0.05). Full summaries also retain every eligible-control contrast with a Bonferroni bound across five controls for each method and regime. No confirmation winner is substituted for LightGBM CQR.

In [5]:
targets = []
for r in rows:
    if r["method"] not in ("baseline_discovery_absolute", "corrected_discovery_absolute",
                            "corrected_discovery_poisson_normalized", "supplied_fixed_poisson_normalized"):
        continue
    available = "width_excess" in r
    targets.append({"Regime": r["panel"], "Method": label(r["method"]),
        "Excess ± SE (%)": f'{r["width_excess"]:.2f} ± {r["width_excess_se"]:.2f}' if available else "unavailable",
        "Upper 95% (%)": f'{r["width_excess_upper95"]:.2f}' if available else "unavailable",
        "Coverage eligible": str(r["coverage_eligible"] and r["reference_eligible"]),
        "Point target": str(r.get("width_point_target", False)),
        "Upper-bound target": str(r.get("width_upper_target", False))})
(GT(pl.DataFrame(targets), groupname_col="Regime")
 .tab_header(title="Real-width target against prespecified LightGBM CQR",
             subtitle="Paired percentage excess; target at most 1%, smaller is better")
 .tab_source_note("SE and Student-t limits use 50 independent draws per regime. Eligibility requires both "
                  "method and reference mean coverage in [0.88, 0.92]. A target passage is scoped to this "
                  "fixed method, regime and comparator, not universal superiority. Source: confirmation_summary.json."))

GT(_tbl_data=shape: (20, 7)
┌───────────────┬────────────┬──────────────────┬───────────┬──────────┬────────┬─────────────┐
│ Regime        ┆ Method     ┆ Excess ± SE (%)  ┆ Upper 95% ┆ Coverage ┆ Point  ┆ Upper-bound │
│ ---           ┆ ---        ┆ ---              ┆ (%)       ┆ eligible ┆ target ┆ target      │
│ str           ┆ str        ┆ str              ┆ ---       ┆ ---      ┆ ---    ┆ ---         │
│               ┆            ┆                  ┆ str       ┆ str      ┆ str    ┆ str         │
╞═══════════════╪════════════╪══════════════════╪═══════════╪══════════╪════════╪═════════════╡
│ low           ┆ Discovery  ┆ 16.35 ± 1.51     ┆ 18.88     ┆ False    ┆ False  ┆ False       │
│               ┆ before /   ┆                  ┆           ┆          ┆        ┆             │
│               ┆ absolute   ┆                  ┆           ┆          ┆        ┆             │
│ low           ┆ Discovery  ┆ 19.80 ± 1.54     ┆ 22.38     ┆ False    ┆ False  ┆ False       │
│               ┆ matched /  ┆                  ┆           ┆          ┆        ┆             │
│               ┆ absolute   ┆                  ┆           ┆          ┆        ┆             │
│ low           ┆ Discovery  ┆ 29.53 ± 2.36     ┆ 33.49     ┆ False    ┆ False  ┆ False       │
│               ┆ matched /  ┆                  ┆           ┆          ┆        ┆             │
│               ┆ normalized ┆                  ┆           ┆          ┆        ┆             │
│ low           ┆ Supplied   ┆ 28.98 ± 1.94     ┆ 32.24     ┆ False    ┆ False  ┆ False       │
│               ┆ fixed /    ┆                  ┆           ┆          ┆        ┆             │
│               ┆ normalized ┆                  ┆           ┆          ┆        ┆             │
│ original      ┆ Discovery  ┆ -1.59 ± 1.68     ┆ 1.23      ┆ True     ┆ True   ┆ False       │
│               ┆ before /   ┆                  ┆           ┆          ┆        ┆             │
│               ┆ absolute   ┆                  ┆           ┆          ┆        ┆             │
│ …             ┆ …          ┆ …                ┆ …         ┆ …        ┆ …      ┆ …           │
│ overdispersed ┆ Supplied   ┆ -3.85 ± 1.25     ┆ -1.76     ┆ True     ┆ True   ┆ True        │
│               ┆ fixed /    ┆                  ┆           ┆          ┆        ┆             │
│               ┆ normalized ┆                  ┆           ┆          ┆        ┆             │
│ zero_inflated ┆ Discovery  ┆ 29.31 ± 0.78     ┆ 30.62     ┆ True     ┆ False  ┆ False       │
│               ┆ before /   ┆                  ┆           ┆          ┆        ┆             │
│               ┆ absolute   ┆                  ┆           ┆          ┆        ┆             │
│ zero_inflated ┆ Discovery  ┆ 31.24 ± 0.95     ┆ 32.82     ┆ True     ┆ False  ┆ False       │
│               ┆ matched /  ┆                  ┆           ┆          ┆        ┆             │
│               ┆ absolute   ┆                  ┆           ┆          ┆        ┆             │
│ zero_inflated ┆ Discovery  ┆ 14.70 ± 0.72     ┆ 15.90     ┆ True     ┆ False  ┆ False       │
│               ┆ matched /  ┆                  ┆           ┆          ┆        ┆             │
│               ┆ normalized ┆                  ┆           ┆          ┆        ┆             │
│ zero_inflated ┆ Supplied   ┆ 18.74 ± 1.65     ┆ 21.51     ┆ True     ┆ False  ┆ False       │
│               ┆ fixed /    ┆                  ┆           ┆          ┆        ┆             │
│               ┆ normalized ┆                  ┆           ┆          ┆        ┆             │
└───────────────┴────────────┴──────────────────┴───────────┴──────────┴────────┴─────────────┘, _body=<great_tables._gt_data.Body object at 0x10d91e990>, _boxhead=Boxhead([ColInfo(var='Regime', type=<ColInfoTypeEnum.row_group: 3>, column_label='Regime', column_align='left', column_width=None), ColInfo(var='Method', type=<ColInfoTypeEnum.default: 1>, column_label='Method', column_align='left', column_width=None), Col

In [6]:
high_old = next(r for r in rows if r["panel"] == "high" and r["method"] == "baseline_discovery_absolute")
high_new = next(r for r in rows if r["panel"] == "high" and r["method"] == "corrected_discovery_absolute")
original_default = next(r for r in rows if r["panel"] == "original" and r["method"] == "corrected_discovery_absolute")
zero_normalized = next(r for r in rows if r["panel"] == "zero_inflated" and r["method"] == "corrected_discovery_poisson_normalized")
high_change = next(r for r in changes if r["panel"] == "high" and r["score"] == "absolute")

(GT(pl.DataFrame([{"Fit": "Before", "Mean MSE": high_old["true_mean_mse"],
                   "Stops equal to one": high_change["baseline_stops"].count(1)},
                  {"Fit": "Matched rate", "Mean MSE": high_new["true_mean_mse"],
                   "Stops equal to one": high_change["corrected_stops"].count(1)}]))
 .tab_header(title="High-rate mean estimation", subtitle="MSE against the true conditional mean; smaller is better")
 .fmt_number(columns="Mean MSE", decimals=5)
 .tab_source_note("Stops describe the 50 actual discovery fits. This compares mean functions; it does not "
                  "identify CQR midpoints as conditional means. Full selected stops and family counts are retained."))

GT(_tbl_data=shape: (2, 3)
┌──────────────┬───────────┬────────────────────┐
│ Fit          ┆ Mean MSE  ┆ Stops equal to one │
│ ---          ┆ ---       ┆ ---                │
│ str          ┆ f64       ┆ i64                │
╞══════════════╪═══════════╪════════════════════╡
│ Before       ┆ 14.364701 ┆ 24                 │
│ Matched rate ┆ 0.271616  ┆ 0                  │
└──────────────┴───────────┴────────────────────┘, _body=<great_tables._gt_data.Body object at 0x122bcc2c0>, _boxhead=Boxhead([ColInfo(var='Fit', type=<ColInfoTypeEnum.default: 1>, column_label='Fit', column_align='left', column_width=None), ColInfo(var='Mean MSE', type=<ColInfoTypeEnum.default: 1>, column_label='Mean MSE', column_align='right', column_width=None), ColInfo(var='Stops equal to one', type=<ColInfoTypeEnum.default: 1>, column_label='Stops equal to one', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x123201fa0>, _spanners=Spanners([]), _heading=Heading(title='High-rate mean estimation', subtitle='MSE against the true conditional mean; smaller is better', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x10d91c590>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x10d91c0e0>, _source_notes=['Stops describe the 50 actual discovery fits. This compares mean functions; it does not identify CQR midpoints as conditional means. Full selected stops and family counts are retained.'], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x122af7bf0>, _formats=[<great_tables._gt_data.FormatInfo object at 0x123200bc0>], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_top_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_right_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_right_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_right_color=OptionsInfo(scss=True, category='table', type='value', value='#D3D3D3'), table_border_bottom_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_bottom_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_bottom_width=OptionsInfo(scss=True, category='tabl

Default high-rate real-width excess changes from {eval}`round(high_old["width_excess"], 2)`% to {eval}`round(high_new["width_excess"], 2)`%, with corrected coverage {eval}`round(high_new["coverage"], 5)`. Its one-sided upper limit is {eval}`round(high_new["width_excess_upper95"], 2)`%. At the original rate, the corrected default has {eval}`round(original_default["width_excess"], 2)`% width excess and does not meet the target. Normalized original-rate rows are useful options, but the unchanged default policy leaves the efficiency question open.

The zero-inflated normalized discovery band retains {eval}`round(zero_normalized["width_excess"], 2)`% real-width excess, despite improved mean error. This is a separate limitation. Central-count quantile scores retain low-count conservatism; their complete ineligible rows remain visible. The high-rate result supports a bounded improvement in the mean fit, rather than a claim that all count regimes are solved.

## Sharp-mean controls

The [control protocol](../appendix/records/spread_and_distributional.md) freezes axis-aligned and shifted pair boundaries, each with constant and genuinely varying scale, plus smooth constant/absolute/bell/monotone, null and heavy-tail controls. There are ten development draws per process, with 1,000 analysis rows, 2,000 independent test rows, and the same 500/500 fitting/calibration split. Sharp means are $3\,\mathrm{sign}(x_1-a)\mathrm{sign}(x_2-b)+x_3$; thresholds are $(0,0)$ or $(0.25,-0.2)$. Constant sharp sd is 0.5; varying sd is $0.3+1.5|x_1|$. Smooth means are $\sin(2\pi x_1)+0.5x_2$.

These are development diagnostics for the mean-approximation question, with no Gaussian spread-rule change and no efficiency confirmation. The supplied controls use fixed GaussianLSS mean specifications and constant/varying sigma. An explicitly labelled oracle subtracts the true mean only for a residual diagnostic; its information never enters discovery.

In [7]:
conditional = pl.DataFrame([{
    "Regime": r["panel"], "Method": label(r["method"]),
    "Marginal": r.get("coverage"), "Low scale": r.get("coverage_low"),
    "High scale": r.get("coverage_high"), "Failures": r["failures"]} for r in rows
    if r["method"] in ("baseline_discovery_absolute", "corrected_discovery_absolute",
                        "corrected_discovery_poisson_normalized", "lightgbm_cqr")])
(GT(conditional, groupname_col="Regime")
 .tab_header(title="Coverage by true-scale third", subtitle="Means over 50 draws; nominal marginal coverage 0.90")
 .fmt_number(columns=["Marginal", "Low scale", "High scale"], decimals=4)
 .sub_missing(missing_text="unavailable")
 .tab_source_note("Low/high scale select test rows in the lower/upper third of the generating standard deviation "
                  "within each draw. Their coverage is descriptive; marginal conformal validity does not guarantee "
                  "90% coverage in each third. A failed method has no full-draw estimate. Source: confirmation_summary.json."))

GT(_tbl_data=shape: (20, 6)
┌───────────────┬────────────────────────────────┬──────────┬───────────┬────────────┬──────────┐
│ Regime        ┆ Method                         ┆ Marginal ┆ Low scale ┆ High scale ┆ Failures │
│ ---           ┆ ---                            ┆ ---      ┆ ---       ┆ ---        ┆ ---      │
│ str           ┆ str                            ┆ f64      ┆ f64       ┆ f64        ┆ i64      │
╞═══════════════╪════════════════════════════════╪══════════╪═══════════╪════════════╪══════════╡
│ low           ┆ Discovery before / absolute    ┆ 0.91238  ┆ 0.896647  ┆ 0.922335   ┆ 0        │
│ low           ┆ Discovery matched / absolute   ┆ 0.91276  ┆ 0.897545  ┆ 0.921557   ┆ 0        │
│ low           ┆ Discovery matched / normalized ┆ 0.9084   ┆ 0.894012  ┆ 0.917605   ┆ 0        │
│ low           ┆ lightgbm cqr                   ┆ 0.94322  ┆ 0.963713  ┆ 0.918563   ┆ 0        │
│ original      ┆ Discovery before / absolute    ┆ 0.89612  ┆ 0.966347  ┆ 0.781796   ┆ 0        │
│ …             ┆ …                              ┆ …        ┆ …         ┆ …          ┆ …        │
│ overdispersed ┆ lightgbm cqr                   ┆ 0.90164  ┆ 0.94497   ┆ 0.859341   ┆ 0        │
│ zero_inflated ┆ Discovery before / absolute    ┆ 0.8999   ┆ 0.99994   ┆ 0.707904   ┆ 0        │
│ zero_inflated ┆ Discovery matched / absolute   ┆ 0.90084  ┆ 0.999641  ┆ 0.714251   ┆ 0        │
│ zero_inflated ┆ Discovery matched / normalized ┆ 0.90098  ┆ 0.993772  ┆ 0.733772   ┆ 0        │
│ zero_inflated ┆ lightgbm cqr                   ┆ 0.89964  ┆ 0.918862  ┆ 0.884371   ┆ 0        │
└───────────────┴────────────────────────────────┴──────────┴───────────┴────────────┴──────────┘, _body=<great_tables._gt_data.Body object at 0x123201d30>, _boxhead=Boxhead([ColInfo(var='Regime', type=<ColInfoTypeEnum.row_group: 3>, column_label='Regime', column_align='left', column_width=None), ColInfo(var='Method', type=<ColInfoTypeEnum.default: 1>, column_label='Method', column_align='left', column_width=None), ColInfo(var='Marginal', type=<ColInfoTypeEnum.default: 1>, column_label='Marginal', column_align='right', column_width=None), ColInfo(var='Low scale', type=<ColInfoTypeEnum.default: 1>, column_label='Low scale', column_align='right', column_width=None), ColInfo(var='High scale', type=<ColInfoTypeEnum.default: 1>, column_label='High scale', column_align='right', column_width=None), ColInfo(var='Failures', type=<ColInfoTypeEnum.default: 1>, column_label='Failures', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x123201430>, _spanners=Spanners([]), _heading=Heading(title='Coverage by true-scale third', subtitle='Means over 50 draws; nominal marginal coverage 0.90', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x123202390>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x1232021e0>, _source_notes=['Low/high scale select test rows in the lower/upper third of the generating standard deviation within each draw. Their coverage is descriptive; marginal conformal validity does not guarantee 90% coverage in each third. A failed method has no full-draw estimate. Source: confirmation_summary.json.'], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x1232021b0>, _formats=[<great_tables._gt_data.FormatInfo object at 0x123201490>], _substitutions=[<great_tables._gt_data.FormatInfo object at 0x123201fd0>], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto

In [8]:
control_rows = []
for panel in dict.fromkeys(r["panel"] for r in sharp):
    a = next(r for r in sharp if r["panel"] == panel and r["method"] == "discovery_spread")
    b = next(r for r in sharp if r["panel"] == panel and r["method"] == "discovery_constant")
    control_rows.append({"Process": panel, "True varying scale": str(a["true_varying_scale"]),
        "Scale selected / 10": a["selected_varying_scale"],
        "Spread mean MSE": a["true_mean_mse"], "Constant mean MSE": b["true_mean_mse"],
        "Spread coverage": a["coverage"], "Boundary coverage": a["boundary_coverage"]})
(GT(pl.DataFrame(control_rows))
 .tab_header(title="Mean-adequacy and scale control inventory", subtitle="Ten development draws per process; MSE in squared response units")
 .fmt_number(columns=["Spread mean MSE", "Constant mean MSE", "Spread coverage", "Boundary coverage"], decimals=4)
 .tab_source_note("Scale selected counts discovery's adopted varying-scale models. In constant-variance rows it is "
                  "false mechanism attribution, not necessarily worse prediction. Boundary coverage uses test rows "
                  "within 0.1 of either sharp threshold; it is an interpretation diagnostic on the smooth controls. "
                  "All selected families here are Gaussian. Complete supplied/oracle controls and MC SEs are in "
                  "sharp_mean_controls/summary.json. No confirmation result is claimed."))

GT(_tbl_data=shape: (10, 7)
┌──────────────────┬─────────┬──────────────────┬─────────────────┬──────────┬──────────┬──────────┐
│ Process          ┆ True    ┆ Scale selected / ┆ Spread mean MSE ┆ Constant ┆ Spread   ┆ Boundary │
│ ---              ┆ varying ┆ 10               ┆ ---             ┆ mean MSE ┆ coverage ┆ coverage │
│ str              ┆ scale   ┆ ---              ┆ f64             ┆ ---      ┆ ---      ┆ ---      │
│                  ┆ ---     ┆ i64              ┆                 ┆ f64      ┆ f64      ┆ f64      │
│                  ┆ str     ┆                  ┆                 ┆          ┆          ┆          │
╞══════════════════╪═════════╪══════════════════╪═════════════════╪══════════╪══════════╪══════════╡
│ axis_constant    ┆ False   ┆ 3                ┆ 1.570123        ┆ 0.341575 ┆ 0.89505  ┆ 0.796193 │
│ axis_varying     ┆ True    ┆ 0                ┆ 0.552127        ┆ 0.552127 ┆ 0.8984   ┆ 0.87451  │
│ shifted_constant ┆ False   ┆ 2                ┆ 0.50281         ┆ 0.266807 ┆ 0.89705  ┆ 0.828899 │
│ shifted_varying  ┆ True    ┆ 0                ┆ 0.432056        ┆ 0.432056 ┆ 0.9032   ┆ 0.895268 │
│ smooth_constant  ┆ False   ┆ 0                ┆ 0.035723        ┆ 0.035723 ┆ 0.89625  ┆ 0.901369 │
│ smooth_absolute  ┆ True    ┆ 10               ┆ 0.061527        ┆ 0.057329 ┆ 0.8964   ┆ 0.915941 │
│ smooth_bell      ┆ True    ┆ 10               ┆ 0.020937        ┆ 0.026015 ┆ 0.8986   ┆ 0.897181 │
│ smooth_monotone  ┆ True    ┆ 10               ┆ 0.063171        ┆ 0.05007  ┆ 0.90025  ┆ 0.901064 │
│ null_constant    ┆ False   ┆ 0                ┆ 0.002346        ┆ 0.002346 ┆ 0.8951   ┆ 0.900179 │
│ heavy_tail       ┆ False   ┆ 0                ┆ 0.0644          ┆ 0.0644   ┆ 0.90585  ┆ 0.905966 │
└──────────────────┴─────────┴──────────────────┴─────────────────┴──────────┴──────────┴──────────┘, _body=<great_tables._gt_data.Body object at 0x116c44fe0>, _boxhead=Boxhead([ColInfo(var='Process', type=<ColInfoTypeEnum.default: 1>, column_label='Process', column_align='left', column_width=None), ColInfo(var='True varying scale', type=<ColInfoTypeEnum.default: 1>, column_label='True varying scale', column_align='left', column_width=None), ColInfo(var='Scale selected / 10', type=<ColInfoTypeEnum.default: 1>, column_label='Scale selected / 10', column_align='right', column_width=None), ColInfo(var='Spread mean MSE', type=<ColInfoTypeEnum.default: 1>, column_label='Spread mean MSE', column_align='right', column_width=None), ColInfo(var='Constant mean MSE', type=<ColInfoTypeEnum.default: 1>, column_label='Constant mean MSE', column_align='right', column_width=None), ColInfo(var='Spread coverage', type=<ColInfoTypeEnum.default: 1>, column_label='Spread coverage', column_align='right', column_width=None), ColInfo(var='Boundary coverage', type=<ColInfoTypeEnum.default: 1>, column_label='Boundary coverage', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x1232031d0>, _spanners=Spanners([]), _heading=Heading(title='Mean-adequacy and scale control inventory', subtitle='Ten development draws per process; MSE in squared response units', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x10d91f6b0>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x1231dfe00>, _source_notes=["Scale selected counts discovery's adopted varying-scale models. In constant-variance rows it is false mechanism attribution, not necessarily worse prediction. Boundary coverage uses test rows within 0.1 of either sharp threshold; it is an interpretation diagnostic on the smooth controls. All selected families here are Gaussian. Complete supplied/oracle controls and MC SEs are in sharp_mean_controls/summary.json. No confirmation result is claimed."], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x122d29b50>, _formats=[<great_tables._gt_data.FormatInfo object at 0x123202600>], _substitutions=[], _col_merge=[], _transforms=[], _

The sharp constant-variance controls expose false scale attribution, while their varying-scale counterparts expose missed scale adoption. The smooth scale controls establish that the existing stage can recover simpler variation. Both types of control are needed before a mean-adequacy change is evaluated. The oracle-instrument failures are retained separately: 85 records, including 17 unsupported-keyword errors. All 500 development records succeed, and the successful initial model predictions reproduce within the stated numerical tolerance.

A scale-approximation, stopping and calibration candidate has not yet been specified or compared in full. The original ten-column task has not been retested. A `btree` term already forms successive boosting trees, so the phrase “single joint tree” does not imply an ensemble rewrite.

The held-back sharp-control confirmation seeds were used by the mean-adequacy guard confirmation. On the two sharp varying-scale processes the candidate adopted a varying scale on 0 of 40 draws each (seeds 6294100–6294139), against 40 of 40 on every smooth varying-scale process. A [decision record](../appendix/records/spread_and_distributional.md) accepts that miss as a documented limitation of the default. The decision record states the limitation and the recall figure.

## Stopping ties and evidence

The stopping rule retains the first exact computed minimum on the positive grid. It introduces no tolerance chosen from outcomes. The original flat-curve stopping labels remain in the per-draw archive (a release asset listed in `docs/book/benchmarks/raw_archives.json`); attained risk and prediction differences matter alongside the labels. The one-standard-error rule (stop at the simplest model whose risk is within one standard error of the minimum) is a separate statistical choice. The design of each open item is recorded in `docs/benchmarks/gap_resolution_design.md`.

All five count panels share streams but are summarized separately. Before/after comparisons pair methods within a draw. The six unsupported overdispersed score rows are one selected LSS model in each implementation, each represented by three failed score rows; none is replaced. The instrument conservatively refuses this type and therefore supplies no full-draw discovery estimate for that panel. Supplied count-model and competitor results remain available.

The record below is the corrected high-rate discovery record for seed 6494100, recorded by running the study on that one draw with mboostrs {eval}`facts["example_record"]["mboostrs_version"]`. It has the same input and partition as the archived draw and is the fit computed by the example cell. The archived confirmation run, made with mboostrs 0.1.3, selected `bbs(x1, df=4.0) + bbs(x2, df=4.0)` with 33 updates and an absolute-score correction of 6.358 on this draw; current code selects {eval}`facts["example_record"]["formula"]` with {eval}`facts["example_record"]["mstop"]` updates and a correction of {eval}`round(facts["example_record"]["quantile"], 3)`. The tables summarize the archived 0.1.3 run.

In [9]:
record = facts["example_record"]

# The live example cell runs the same draw, seed and settings; both must describe one fit.
assert (selected.family, selected.formula, selected.mstop) == (record["family"], record["formula"], record["mstop"])
assert np.isclose(band.quantile, record["quantile"], rtol=1e-10)

observed = {"Regime / seed": f'{record["panel"]} / {record["seed"]}',
            "Code version": f'mboostrs {record["mboostrs_version"]}, commit {record["source_commit"][:8]}',
            "Selected family": record["family"], "Selected formula": record["formula"],
            "Stop": record["mstop"], "Fit rate": record["nu"],
            "Observed tuning rates": str(record["tuning_nu"]),
            "Fit / calibration rows": f'{record["n_fit"]} / {record["n_calibration"]}',
            "Correction": record["quantile"]}
(GT(pl.DataFrame([{"Field": k, "Value": str(v)} for k, v in observed.items()]))
 .tab_header(title="One complete confirmation record", subtitle="Actual rates, model choice and calibration evidence")
 .tab_source_note("Fields of the corrected high-rate discovery record for seed 6494100, re-recorded with "
                  "--seeds 6494100 into confirmation_example_high_6494100.jsonl.gz. "
                  "benches/summarize_poisson_tuning_consistency.py checks its hashes and that its input and "
                  "partition equal the archived draw, then copies these fields into confirmation_validation.json. "
                  "The full record also contains test truth, endpoint/point arrays, input hash, row partitions, "
                  "tuning risk curves, warnings and provenance."))

GT(_tbl_data=shape: (9, 2)
┌────────────────────────┬─────────────────────────────────┐
│ Field                  ┆ Value                           │
│ ---                    ┆ ---                             │
│ str                    ┆ str                             │
╞════════════════════════╪═════════════════════════════════╡
│ Regime / seed          ┆ high / 6494100                  │
│ Code version           ┆ mboostrs 0.2.0, commit ac8141b… │
│ Selected family        ┆ Poisson                         │
│ Selected formula       ┆ bols(x1) + bols(x2)             │
│ Stop                   ┆ 145                             │
│ Fit rate               ┆ 0.02                            │
│ Observed tuning rates  ┆ [0.02, 0.02]                    │
│ Fit / calibration rows ┆ 500 / 500                       │
│ Correction             ┆ 6.305047936933793               │
└────────────────────────┴─────────────────────────────────┘, _body=<great_tables._gt_data.Body object at 0x123203a40>, _boxhead=Boxhead([ColInfo(var='Field', type=<ColInfoTypeEnum.default: 1>, column_label='Field', column_align='left', column_width=None), ColInfo(var='Value', type=<ColInfoTypeEnum.default: 1>, column_label='Value', column_align='left', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x112fd0ec0>, _spanners=Spanners([]), _heading=Heading(title='One complete confirmation record', subtitle='Actual rates, model choice and calibration evidence', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x123203ad0>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x1232039e0>, _source_notes=['Fields of the corrected high-rate discovery record for seed 6494100, re-recorded with --seeds 6494100 into confirmation_example_high_6494100.jsonl.gz. benches/summarize_poisson_tuning_consistency.py checks its hashes and that its input and partition equal the archived draw, then copies these fields into confirmation_validation.json. The full record also contains test truth, endpoint/point arrays, input hash, row partitions, tuning risk curves, warnings and provenance.'], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x1232020f0>, _formats=[], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_top_color=OptionsInfo(scs

## Discovery stopping ceilings

The sharp-mean controls use the public default ceilings: 400 updates for the ordinary mean search and 1,600 updates per parameter for an adopted Gaussian location-scale search. The [budget clarification](default_stopping_ceilings.md) records the units of the stopping counts. 

## Reproduction

The [protocol](poisson_tuning_consistency/PROTOCOL.md) fixes the design. The baseline package runs from its immutable checkout; the current and corrected phases use the identical native release extension and record loaded Python source hashes independently. No count default or Gaussian spread policy changed during measurement.

Run the recorded study from the repository environment with its matching native extension:

```bash
PYTHONPATH=python:. .venv/bin/python benches/study_poisson_tuning_consistency.py \
  --phase confirmation --variant corrected --output /tmp/count-confirmation.jsonl.gz
PYTHONPATH=python:. .venv/bin/python benches/study_poisson_tuning_consistency.py \
  --phase confirmation --variant corrected --panels high --seeds 6494100 \
  --output docs/book/benchmarks/poisson_tuning_consistency/confirmation_example_high_6494100.jsonl.gz
PYTHONPATH=python:. .venv/bin/python -m benches.summarize_poisson_tuning_consistency
PYTHONPATH=python:. .venv/bin/python -m benches.summarize_sharp_mean_controls
PYTHONPATH=python:. .venv/bin/python -m benches.verify_gap_predictions
```

Runners refuse to overwrite completed or partial evidence. The publication archives use XZ compression to retain a longer compression window; their decompressed JSONL bytes are identical to the measured gzip output. Manifests retain both the original gzip digest and the published compressed/uncompressed digests, and phase metadata is unchanged. The protocol, run metadata, archive manifests (`archives.json`), summary files (`development_summary.json`, `confirmation_summary.json`) and validation files accompany the notebook download. The per-draw archives are assets of the `benchmark-raw-archives-1` GitHub release; `docs/book/benchmarks/raw_archives.json` lists each file with its size, SHA-256 and the command that regenerates it. Workflow times are descriptive: panel jobs ran concurrently, each model used one thread, and the study does not claim a controlled speed comparison.